<a href="https://colab.research.google.com/github/leejuheon06/Practice_ML_1/blob/main/08_MNIST%EB%A5%BC_%ED%99%9C%EC%9A%A9%ED%95%9C_%EC%88%AB%EC%9E%90_%EC%9D%B8%EC%8B%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt-get install -y fonts-nanum* | tail -n 1
!sudo fc-cache -fv
!rm -rf ~/.cache/matplotlib

In [ ]:
# 필요 라이브러리 설치

!pip install torchviz | tail -n 1
!pip install torchinfo | tail -n 1

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
from torchviz import make_dot
import seaborn as sns
from sklearn.datasets import load_digits  # 손글씨 숫자 데이터 (0~9)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

print(f"PyTorch 버전: {torch.__version__}")
print(f"NumPy 버전: {np.__version__}")

In [ ]:
# 폰트 관련 용도
import matplotlib.font_manager as fm

# 나눔 고딕 폰트의 경로 명시
path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_name = fm.FontProperties(fname=path, size=10).get_name()

In [ ]:
# 기본 폰트 설정
plt.rcParams['font.family'] = font_name

# 기본 폰트 사이즈 변경
plt.rcParams['font.size'] = 14

# 기본 그래프 사이즈 변경
plt.rcParams['figure.figsize'] = (6,6)

# 기본 그리드 표시
# 필요에 따라 설정할 때는, plt.grid()
plt.rcParams['axes.grid'] = True

# 마이너스 기호 정상 출력
plt.rcParams['axes.unicode_minus'] = False

# 넘파이 부동소수점 자릿수 표시
np.set_printoptions(suppress=True, precision=4)

8.4 활성화 함수와 ReLU 함수

In [ ]:
# ReLU 함수의 그래프

relu = nn.ReLU()
x_np = np.arange(-2, 2.1, 0.25)
x = torch.tensor(x_np).float()
y = relu(x)

plt.plot(x.detach(), y.detach())
plt.title('ReLU')
plt.show()

8.5 GPU 사용하기

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# 텐서 변수 x, y
x_np = np.arange(-2.0, 2.1, 0.25)
y_np = np.arange(-1.0, 3.1, 0.25)
x = torch.tensor(x_np)
y = torch.tensor(y_np)

# x와 y 연산
z = x * y
print(z)

In [ ]:
# 변수 GPU로 보내기
x = x.to(device)

# device 속성 확인
print(x.device)
print(y.device)

In [ ]:
# z = x * y
# Error 발생!
# 다른 device 위치한 데이터 간 연산 불가!

# RuntimeError                              Traceback (most recent call last)
# /tmp/ipykernel_3944/880245365.py in <cell line: 0>()
# ----> 1 z = x * y

# RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

In [ ]:
y = y.to(device)

z = x * y
print(z)

8.8 데이터 준비 1 (Dataset을 활용해 불러오기)

In [ ]:
# 라이브러리 임포트
import torchvision.datasets as datasets

# 다운로드받을 디렉터리명
data_root = './data'

train_set0 = datasets.MNIST(
    # 원본 데이터를 다운로드 받을 디렉터리 지정
    root = data_root,
    # 훈련 데이터인지 검증 데이터인지 설정
    train = True,
    # 원본 데이터가 없는 경우, 다운로로드를 실행하는지 여부
    download = True
)

In [ ]:
!ls -lR ./data/MNIST

In [ ]:
# 데이터 건수 확인
print('데이터 건수:', len(train_set0))

# 첫번째 요소 가져오기
image, label = train_set0[0]

# 데이터 타입 확인
print('입력 데이터 타입:', type(image))
print('입력 데이터 타입:', type(label))

In [ ]:
# 입력 데이터를 이미지로 출력
plt.figure(figsize=(1,1))
plt.title(f'{label}')
plt.imshow(image, cmap='gray_r')
plt.axis('off')
plt.show()

In [ ]:
# 정답 데이터와 함께 처음 20개 데이터를 이미지로 출력
plt.figure(figsize=(10,3))
for i in range(20):
    ax = plt.subplot(2, 10, i+1)

    # image, label 취득
    image, label = train_set0[i]

    # 이미지 출력
    plt.imshow(image, cmap='gray_r')
    ax.set_title(f'{label}')
    ax.get_xaxis().set_visible(False)   # 이미지 x 좌표축 표기 여부
    ax.get_yaxis().set_visible(False)   # 이미지 y 좌표축 표기 여부
plt.show()

8.9 데이터 준비 2 (Transforms를 활용한 데이터 전처리)

In [ ]:
# 라이브러리 임포트
import torchvision.transforms as transforms

transfrom1 = transforms.Compose([
    # 데이터를 텐서로 변환
    transforms.ToTensor()
])

train_set1 = datasets.MNIST(
    root=data_root,
    train=True,
    download=True,
    transform=transfrom1
)

In [ ]:
# 변환 결과 확인
image, label = train_set1[0]
print('입력 데이터 타입:', type(image))
print('입력 데이터 크기:', image.shape)
print('입력 데이터 최솟값:', image.min())
print('입력 데이터 최댓값:', image.max())

In [ ]:
transform2 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0.5,0.5)
])

train_set2 = datasets.MNIST(
    root=data_root,
    train=True,
    download=True,
    transform=transform2
)

In [ ]:
# 변환 결과 확인
image, label = train_set2[0]
print('입력 데이터 타입:', type(image))
print('입력 데이터 크기:', image.shape)
print('입력 데이터 최솟값:', image.min())
print('입력 데이터 최댓값:', image.max())

In [ ]:
transform3 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0.5,0.5),
    transforms.Lambda(lambda x : x.view(-1))
])

train_set3 = datasets.MNIST(
    root=data_root,
    train=True,
    download=True,
    transform=transform3
)

In [ ]:
# 변환 결과 확인
image, label = train_set3[0]
print('입력 데이터 타입:', type(image))
print('입력 데이터 크기:', image.shape)
print('입력 데이터 최솟값:', image.min())
print('입력 데이터 최댓값:', image.max())

In [ ]:
# 데이터 변환용 함수 Transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0.5,0.5),
    transforms.Lambda(lambda x : x.view(-1))
])

In [ ]:
# 데이터 취득용 함수 Dataset

# 훈련 데이터셋 정의
train_set = datasets.MNIST(
    root=data_root,
    train=True,
    download=True,
    transform=transform
)

# 검증 데이터셋 정의
test_set = datasets.MNIST(
    root=data_root,
    train=False,
    download=True,
    transform=transform
)

8.10 데이터 준비 3 (데이터로더를 활용한 미니 배치 데이터 생성)

In [ ]:
from torch.utils.data import DataLoader

# 미니 배치 사이즈 지정
batch_size = 500

# 훈련용 데이터 로더
train_loader = DataLoader(
    train_set, batch_size=batch_size, shuffle=True
)

# 검증용 데이터 로더
test_loader = DataLoader(
    test_set, batch_size=batch_size, shuffle=False
)

In [ ]:
# 몇 개의 그룹으로 데이터를 가져올 수 있는가
print(len(train_loader))

# 데이터로더로부터 가장 처음 한 세트를 가져옴
for images, labels in train_loader:
    break

print(images.shape)
print(labels.shape)

8.11 모델 정의

In [ ]:
# 입력 차원수
n_input = image.shape[0]

# 출력 차원수 (분류 클래스 수는 10)
n_output = len(set(list(labels.data.numpy())))

# 은닉층의 노드 수
n_hidden = 128

# 결과 확인
print(f'n_input: {n_input}, n_output: {n_output}, n_hidden: {n_hidden}')

In [ ]:
# 모델 정의
class Net(nn.Module):
    def __init__(self, n_input, n_output, n_hidden):
        super().__init__()

        self.l1 = nn.Linear(n_input, n_hidden)
        self.l2 = nn.Linear(n_hidden, n_output)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x1 = self.l1(x)
        x2 = self.relu(x1)
        x3 = self.l2(x2)
        return x3

In [ ]:
# 난수 고정
torch.manual_seed(123)
torch.cuda.manual_seed(123)

# 모델 인스턴스 생성
net = Net(n_input, n_output, n_hidden)

# GPU 전송
net = net.to(device)

In [ ]:
# 학습률
lr = 0.01

# 손실함수
criterion = nn.CrossEntropyLoss()

# 최적화 : 경사하강법
optimizer = optim.SGD(net.parameters(), lr=lr)

# 반복횟수
num_epochs = 30

# 평가 결과 기록
history = np.zeros((0,5))

In [ ]:
for parameter in net.named_parameters():
    print(parameter)

In [ ]:
print(net)

8.12 경사 하강법

In [ ]:
# 훈련 데이터셋의 가장 처음 항목을 취득
for images, labels in train_loader:
    break

In [ ]:
# 데이터로더에서 취득한 데이터를 GPU로 보냄
inputs = images.to(device)
labels = labels.to(device)

In [ ]:
outputs = net(inputs)
print(outputs)

In [ ]:
loss = criterion(outputs, labels)

print(loss.item())

# 손실 그래프 시각화
make_dot(loss, params=dict(net.named_parameters()))

In [ ]:
from tqdm.notebook import tqdm

# 반복 계산 메인 루프
for epoch in range(num_epochs):
    train_acc, train_loss = 0, 0
    val_acc, val_loss = 0, 0
    n_train, n_test = 0, 0

    # 훈련 페이즈
    for inputs, labels in tqdm(train_loader):
        n_train += len(labels)

        # GPU 전송
        inputs = inputs.to(device)
        labels = labels.to(device)

        # 경사 초기화
        optimizer.zero_grad()

        # 예측 계산
        outputs = net(inputs)

        # 손실 계산
        loss = criterion(outputs, labels)

        # 경사 계산
        loss.backward()

        # 파라미터 수정
        optimizer.step()

        train_loss += loss.item() # 배치당 손실 누적
        predicted = torch.max(outputs, 1)[1] # 예측값 계산
        train_acc += (predicted == labels).sum().item() # 맞춘 개수 누적

    # 예측 페이즈
    for inputs_test, labels_test in test_loader:
        n_test += len(labels_test)

        inputs_test = inputs_test.to(device)
        labels_test = labels_test.to(device)

        # 예측 계산
        outputs_test = net(inputs_test)

        # 손실계산
        loss_test = criterion(outputs_test, labels_test)

        # 예측값 취득 >> [0]: 최댓값 [1]: 최댓값 위치
        predicted_test = torch.max(outputs_test, 1)[1]

        # 손실과 정확도 계산
        val_loss += loss_test.item()
        val_acc += (predicted_test == labels_test).sum().item()

    # 평균값 계산
    train_loss /= len(train_loader)
    train_acc /= n_train
    val_loss /= len(test_loader)
    val_acc /= n_test

    if ((epoch) % 5 ==0):
        print(f'Epoch [{epoch+1}/{num_epochs}], loss:{train_loss:.5f} acc:{train_acc:.5f} val_loss:{val_loss:.5f} val_acc:{val_acc:.5f}')
        item = np.array([epoch, train_loss, train_acc, val_loss, val_acc])
        history = np.vstack((history, item))


In [ ]:
# 손실과 정확도 확인
print(f'초기상태 : 손실 : {history[0,3]:.5f} 정확도 : {history[0,4]:.5f}')
print(f'최종상태 : 손실 : {history[-1,3]:.5f} 정확도 : {history[-1,4]:.5f}')

In [ ]:
# 학습 곡선 출력(손실)

plt.plot(history[:,0], history[:,1], c='b', label='훈련')
plt.plot(history[:,0], history[:,3], c='k', label='검증')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()
plt.show()

In [ ]:
# 학습 곡선 출력(정확도)

plt.plot(history[:,0], history[:,2], c='b', label='훈련')
plt.plot(history[:,0], history[:,4], c='k', label='검증')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve')
plt.legend()
plt.show()

8.14 은닉층 추가하기

In [ ]:
# 모델 정의
# 784입력 10출력을 갖는 2개의 은닉층을 포함한 신경망

class Net(nn.Module):
    def __init__(self, n_input, n_output, n_hidden):
        super().__init__()

        self.l1 = nn.Linear(n_input, n_hidden)
        self.l2 = nn.Linear(n_hidden, n_hidden)
        self.l3 = nn.Linear(n_hidden, n_output)

        self.relu = nn.ReLU(inplace=True)

        def forward(self, x):
            x1 = self.l1(x)
            x2 = self.relu(x1)
            x3 = self.l2(x2)
            x4 = self.relu(x3)
            x5 = self.l3(x4)
            return x5

In [ ]:
print(net)

In [ ]:
# eos